In [ ]:
import ee
import geemap
import math
from utils.variables import PROJECT, ANALYSIS_END_YR, GLC_LABELS, DRIVER_LABELS

from absolute_effectiveness.site_selector import SiteSelector
from absolute_effectiveness.data_processor import DataProcessor
from absolute_effectiveness.habitat_condition import HabitatConditionAnalyzer
from absolute_effectiveness.habitat_loss import HabitatLossAnalyzer
from absolute_effectiveness.visualization import VisualizationService

ee.Authenticate()
ee.Initialize(project=PROJECT)

site_selector = SiteSelector()
processor = DataProcessor.from_gee_defaults()
condition_analyzer = HabitatConditionAnalyzer()
loss_analyzer = HabitatLossAnalyzer()
visualization_service = VisualizationService()

In [ ]:
# Select site by WDPAID (one from the list of 30 in variables.py)
site_id = 352159

# Get test sites and site-specific information
test_sites = site_selector.get_test_sites()
START_YR = site_selector.set_start_yr(test_sites, site_id)
site_geom = site_selector.get_site_geom(test_sites, site_id)

# Check if PA designation year is valid for analysis
site_selector.check_start_yr(START_YR)

In [ ]:
# Process all input datasets
GLC_processed = processor.process_glc(test_sites, START_YR)
GPW_processed = processor.process_gpw(START_YR)
NFW_processed = processor.process_nfw(test_sites)
HGFC_processed = processor.process_hgfc(START_YR)

# Get habitat raster
habitat_raster = condition_analyzer.get_habitat_raster(
    GLC_processed, HGFC_processed, GPW_processed, NFW_processed
)

In [ ]:
# TEST ALTERNATIVE INTACTNESS CALCULATION
# Build an exponentially decaying distance-weighted kernel

# Get habitat binary
habitat_binary = habitat_raster.gt(0).unmask(0).toFloat()

# Define parameters for kernel
beta = 0.01  # controls the rate of exponential decay
kernel_radius_meters = 2000

pixelSize = 30
kernel_radius_pixels = math.ceil(kernel_radius_meters / pixelSize)
kernel_size = kernel_radius_pixels * 2 + 1  # width and height of the kernel

# Create 2-D array of weights for kernel
weights = []
for row in range(kernel_size):
    row_weights = []
    for col in range(kernel_size):
        dx = (
            col - kernel_radius_pixels
        ) * pixelSize  # horizontal distance from center pixel
        dy = (
            row - kernel_radius_pixels
        ) * pixelSize  # vertical distance from center pixel
        d = math.sqrt(dx * dx + dy * dy)  # hypotenuse distance from center pixel
        w = (
            math.exp(-beta * d) if d <= kernel_radius_meters else 0
        )  # apply exponential decay and clip to kernel radius
        row_weights.append(w)
    weights.append(row_weights)

# Build custom kernel with weights
exp_kernel = ee.Kernel.fixed(
    width=kernel_size,
    height=kernel_size,
    weights=weights,
    normalize=True,  # sum of all weights is 1; this normalizes the intactness values to be between 0 and 1
)

# Apply kernel to habitat binary to get intactness raster
intactness = (
    habitat_binary.convolve(
        exp_kernel
    )  # sum the weighted habitat binary values in the kernel
    .rename("intactness")
    .updateMask(habitat_binary)
)  # only habitat pixels get intactness values

In [ ]:
# VISUALIZE RESULTS OF TESTING ALTERNATIVE INTACTNESS CALCULATION

Map = geemap.Map()

Map.add_basemap("CartoDB.DarkMatter")

Map.addLayer(habitat_binary, {"min": 0, "max": 1}, "Habitat binary")
Map.addLayer(
    intactness.clip(site_geom).reproject(crs=habitat_binary.projection(), scale=30),
    {
        "min": 0,
        "max": 1,
        "palette": ["#d73027", "#fdae61", "#fee08b", "#d9ef8b", "#1a9850"],
    },
    "Intactness",
)

Map.centerObject(site_geom)
Map

In [ ]:
# Process all input datasets
GLC_processed = processor.process_glc(test_sites, START_YR)
GPW_processed = processor.process_gpw(START_YR)
NFW_processed = processor.process_nfw(test_sites)
HGFC_processed = processor.process_hgfc(START_YR)

# Calculate Habitat Extent score
habitat_raster = condition_analyzer.get_habitat_raster(
    GLC_processed, HGFC_processed, GPW_processed, NFW_processed
)
habitat_extent_score = condition_analyzer.calc_habitat_extent_score(
    habitat_raster, site_geom
)

# Calculate Habitat Intactness score
edge_distance_raster = condition_analyzer.get_edge_distance_raster(
    habitat_raster, site_geom
)
edge_distance_score = condition_analyzer.calc_edge_distance_score(
    edge_distance_raster, site_geom
)
patch_size_raster = condition_analyzer.get_patch_size_raster(habitat_raster, site_geom)
patch_size_score = condition_analyzer.calc_patch_size_score(
    patch_size_raster, site_geom
)
habitat_intactness_score = condition_analyzer.calc_intactness_score(
    edge_distance_score, patch_size_score
)

# Calculate overall Habitat Condition score
habitat_condition_score = condition_analyzer.calc_habitat_condition_score(
    habitat_extent_score, habitat_intactness_score
)

# Calculate Habitat Loss score
habitat_loss_raster, habitat_start_raster = loss_analyzer.get_habitat_loss_raster(
    GLC_processed, GPW_processed, HGFC_processed, START_YR
)
habitat_loss_score = loss_analyzer.calc_habitat_loss_score(
    habitat_loss_raster, habitat_start_raster, site_geom
)

# Analyze drivers and types of habitat loss
driver_class = loss_analyzer.get_driver_class_image(
    GLC_processed, GPW_processed, habitat_loss_raster
)
driver_dict = loss_analyzer.calc_class_area_and_pct(driver_class, site_geom)
habitat_class = loss_analyzer.get_habitat_class_image(
    GLC_processed, habitat_loss_raster, START_YR
)
habitat_dict = loss_analyzer.calc_class_area_and_pct(habitat_class, site_geom)

# Create Sentinel-2 composites (for visualization purposes only)
start_s2_composite = visualization_service.get_s2_med_composite(site_geom, START_YR)
end_s2_composite = visualization_service.get_s2_med_composite(
    site_geom, ANALYSIS_END_YR
)

# Print all results
print(f"Site ID: {site_id}")
print(f"Analysis Period: {START_YR} - {ANALYSIS_END_YR}")
print(f"\nHabitat Extent Score: {habitat_extent_score:.4f}")
print(f"Edge Distance Score: {edge_distance_score:.4f}")
print(f"Patch Size Score: {patch_size_score:.4f}")
print(f"Habitat Intactness Score: {habitat_intactness_score:.4f}")
print(f"Habitat Condition Score: {habitat_condition_score:.4f}")
print(f"Habitat Loss Score: {habitat_loss_score:.4f}")
print("\nDrivers of habitat loss:")
loss_analyzer.translate_results(driver_dict, DRIVER_LABELS)
print("\nTop 4 types of habitat lost:")
loss_analyzer.translate_results(habitat_dict, GLC_LABELS)

In [ ]:
# ============================================================================
# VISUALIZATION
# ============================================================================

# from utils.variables import (
#     GLC_PALETTE,
#     MAX_EDGE_DIST,
#     MAX_PATCH_SIZE
# )
# from math import sqrt

# Sentinel-2 RGB viz parameters
s2_rgb_viz = {
    "min": 0.0,
    "max": 0.3,
    "bands": ["B4", "B3", "B2"],
}

Map = geemap.Map()
Map.add_basemap("Esri.WorldImagery")

# Map.addLayer(start_s2_composite, s2_rgb_viz, "S2, Start Year", 0)
# Map.addLayer(end_s2_composite, s2_rgb_viz, "S2, End Year", 0)
# Map.addLayer(GLC_processed.select("GLC_2022"), {"min": 0, "max": 36, "palette": GLC_PALETTE}, "Global Land Cover", 0)
# Map.addLayer(GPW_processed.select("GPW_2022").selfMask(), {"min": 1, "max": 2, "palette": ["#ffcd73", "#ff9916"]}, "Global Pasture Watch", 0)
# Map.addLayer(NFW_processed.selfMask(), {"palette": "teal"}, "Natural Forests of the World", 0)
# Map.addLayer(HGFC_processed.selfMask(), {"min": START_YR - 2000, "max": ANALYSIS_END_YR - 2000, "palette": ["yellow", "red"]}, "Hansen Global Forest Change", 0)
# Map.addLayer(habitat_raster.clip(site_geom), {"palette": GLC_PALETTE}, "Habitat extent")
# Map.addLayer(edge_distance_raster, {"min": 0, "max": MAX_EDGE_DIST, "palette": ["red", "white"]}, "Edge distance", 0)
# Map.addLayer(patch_size_raster.reproject(crs=habitat_raster.projection(), scale=sqrt((MAX_PATCH_SIZE * 1000000) / 1024)), {'min':1, 'max':1024, 'palette': ['red', 'white']}, 'Patch size')
# Map.addLayer(habitat_start_raster.selfMask().clip(site_geom), {'palette': GLC_PALETTE}, "Habitat at start year")
# Map.addLayer(habitat_loss_raster.selfMask().clip(site_geom), {'palette': ['orange']}, "Habitat loss (binary)")
# Map.addLayer(driver_class.selfMask(), {"min": 1, "max": 4, "palette": ["yellow", "red", "orange", "green"]}, "Drivers of habitat loss")
# Map.addLayer(habitat_class.selfMask(), {"palette": GLC_PALETTE}, "Types of habitat lost")
# Map.addLayer(site_geom, {"color": "yellow"}, "Test site")

Map.centerObject(site_geom)

Map